# Forecasting Final Project — Extended Fundamentals Diagnostics
## S&P 500 Energy Sector Index (S5ENRS) — Sample Effect and Variable Selection Audit
### Econ 409 | University of California, Los Angeles
Jacob Williams · Praus Paek · Joshua Kenworthy · Agnibha Bhattacharya · Ignacio Ramirez

---

## Purpose

This notebook diagnoses why adding WTI crude oil and the high-yield credit spread worsened forecasting performance in the first extension analysis. The main confounds are:

1. **Sample truncation.** Including high-yield spread restricts the usable history to post-1997, cutting approximately 6.5 years from the training set. Any performance change may reflect this shortened sample rather than the new variables.
2. **Rigid pre-screening.** The original extension selected feature specifications using one fixed rolling-regression configuration before tuning, which may have misranked specifications that benefit from different window lengths or noise parameters.
3. **Shared feature selection.** Both strategies were forced to use the same selected feature set, preventing each strategy from finding its own best specification.
4. **Transformation assumptions.** WTI and high-yield spread were used in one form only; alternative transformations were not evaluated.

This notebook addresses each issue in turn:
- Constructs two comparison samples (full baseline, restricted common) to isolate the sample-truncation effect
- Tests multiple transformations for each new variable
- Separately selects the best feature specification for the rolling regression and Kalman strategies
- Uses joint feature-and-hyperparameter search rather than pre-screening
- Runs walk-forward backtests for five model families to produce a clean diagnostic comparison

---

## Leakage Prevention

All constraints from the original analysis apply. In addition: feature specification selection, transformation selection, and hyperparameter tuning use training data only. The test period is not examined at any stage during model development.

---

## 1. Data Loading

The original five series are loaded identically to the main project notebook. WTI crude and high-yield spread are loaded from the pre-saved CSVs in `potential fundamentals/`. If those files are absent, the cell below downloads and saves them before loading.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
import os
if not (os.path.exists('potential fundamentals/wti_crude.csv') and
        os.path.exists('potential fundamentals/high_yield_credit_spread.csv')):
    import pandas_datareader.data as web
    import datetime
    os.makedirs('potential fundamentals', exist_ok=True)
    _s = datetime.date(1985, 1, 1)
    _e = datetime.date(2026, 3, 16)
    _wti = web.DataReader('DCOILWTICO',  'fred', _s, _e)
    _wti.index.name = 'Date'; _wti.columns = ['wti_price']
    _wti.to_csv('potential fundamentals/wti_crude.csv')
    _hy = web.DataReader('BAMLH0A0HYM2', 'fred', _s, _e)
    _hy.index.name = 'Date'; _hy.columns = ['hy_spread']
    _hy.to_csv('potential fundamentals/high_yield_credit_spread.csv')

In [ ]:
enrs = pd.read_csv('S5ENGS.csv')
enrs.columns = enrs.columns.str.strip()
enrs = enrs.rename(columns={'S5ENRS Index - Last Price': 'enrs'})
enrs['date'] = pd.to_datetime(enrs['Date'], format='%m/%d/%Y', errors='coerce')
enrs['enrs'] = pd.to_numeric(enrs['enrs'], errors='coerce')
enrs = enrs[['date', 'enrs']].dropna().sort_values('date').reset_index(drop=True)

t1y = pd.read_csv('SP 500 and US Treasury/H15T1Y.csv')
t1y.columns = t1y.columns.str.strip()
t1y = t1y.rename(columns={'Last Price': 't1y'})
t1y['date'] = pd.to_datetime(t1y['Date'], format='%d/%m/%Y', errors='coerce')
t1y['t1y'] = pd.to_numeric(t1y['t1y'], errors='coerce')
t1y = t1y[['date', 't1y']].dropna().sort_values('date').reset_index(drop=True)

bpe = pd.read_csv('SP 500 and US Treasury/SP500 Best PE.csv')
bpe.columns = bpe.columns.str.strip()
bpe = bpe.rename(columns={'BEst P/E Ratio': 'sp500_best_pe'})
bpe['date'] = pd.to_datetime(bpe['Date'], format='%d/%m/%Y', errors='coerce')
bpe['sp500_best_pe'] = pd.to_numeric(bpe['sp500_best_pe'], errors='coerce')
bpe = bpe[['date', 'sp500_best_pe']].dropna().sort_values('date').reset_index(drop=True)

pe = pd.read_csv('SP 500 and US Treasury/SP500 PE.csv')
pe.columns = pe.columns.str.strip()
pe = pe.rename(columns={'Price Earnings Ratio (P/E)': 'sp500_pe'})
pe['date'] = pd.to_datetime(pe['Date'], format='%d/%m/%Y', errors='coerce')
pe['sp500_pe'] = pd.to_numeric(pe['sp500_pe'], errors='coerce')
pe = pe[['date', 'sp500_pe']].dropna().sort_values('date').reset_index(drop=True)

sp500 = pd.read_csv('SP 500 and US Treasury/SP500 Price.csv')
sp500 = sp500.dropna(axis=1, how='all')
sp500.columns = sp500.columns.str.strip()
sp500 = sp500.rename(columns={'Last Price': 'sp500_price'})
sp500['date'] = pd.to_datetime(sp500['Date'], format='%d/%m/%Y', errors='coerce')
sp500['sp500_price'] = pd.to_numeric(sp500['sp500_price'], errors='coerce')
sp500 = sp500[['date', 'sp500_price']].dropna().sort_values('date').reset_index(drop=True)

In [ ]:
wti_df = pd.read_csv('potential fundamentals/wti_crude.csv')
wti_df['date'] = pd.to_datetime(wti_df['Date'])
wti_df['wti_price'] = pd.to_numeric(wti_df['wti_price'], errors='coerce')
wti_df = wti_df[['date', 'wti_price']].dropna().sort_values('date').reset_index(drop=True)

hy_df = pd.read_csv('potential fundamentals/high_yield_credit_spread.csv')
hy_df['date'] = pd.to_datetime(hy_df['Date'])
hy_df['hy_spread'] = pd.to_numeric(hy_df['hy_spread'], errors='coerce')
hy_df = hy_df[['date', 'hy_spread']].dropna().sort_values('date').reset_index(drop=True)

In [ ]:
daily = (
    enrs.set_index('date')
    .join(t1y.set_index('date'),   how='outer')
    .join(bpe.set_index('date'),   how='outer')
    .join(pe.set_index('date'),    how='outer')
    .join(sp500.set_index('date'), how='outer')
    .join(wti_df.set_index('date'), how='outer')
    .join(hy_df.set_index('date'),  how='outer')
    .sort_index()
)
display(daily.shape)
display(daily.tail(3))

---

## 2. Phase 1 — Sample Comparison

Two comparison samples are constructed to isolate the effect of sample truncation from the effect of new variables.

**Full baseline sample** uses the four original signal variables (`enrs`, `t1y`, `sp500_pe`, `sp500_best_pe`). The overlap begins when all four are available (~May 1990).

**Common extended sample** additionally requires `wti_price` and `hy_spread` to be non-null. Because `hy_spread` (BAMLH0A0HYM2) begins 1996-12-31, the overlap starts approximately January 1997, shortening the training set by roughly 6.5 years.

The table below quantifies this loss. The model comparison in Phase 7 uses both samples to separate the sample-truncation effect from the new-variable effect.

In [ ]:
weekly_all = daily.resample('W-FRI').last()

weekly_all['wti_log']  = np.log(weekly_all['wti_price'])
weekly_all['wti_ret']  = weekly_all['wti_price'].pct_change()
weekly_all['hy_level'] = weekly_all['hy_spread'].copy()
weekly_all['hy_chg']   = weekly_all['hy_spread'].diff(1)
weekly_all['enrs_ret'] = weekly_all['enrs'].pct_change()
weekly_all['target']   = weekly_all['enrs_ret'].shift(-1)
weekly_all['sp500_ret']    = weekly_all['sp500_price'].pct_change()
weekly_all['sp500_target'] = weekly_all['sp500_ret'].shift(-1)

FULL_COLS   = ['enrs', 't1y', 'sp500_pe']
COMMON_COLS = ['enrs', 't1y', 'sp500_pe', 'wti_log', 'wti_ret', 'hy_level', 'hy_chg']

weekly_full   = weekly_all[weekly_all[FULL_COLS].notna().all(axis=1)].copy()
weekly_common = weekly_all[weekly_all[COMMON_COLS].notna().all(axis=1)].copy()

SPLIT_DATE = pd.Timestamp('2018-01-05')

train_full   = weekly_full.loc[weekly_full.index < SPLIT_DATE].copy()
test_full    = weekly_full.loc[weekly_full.index >= SPLIT_DATE].copy()
train_common = weekly_common.loc[weekly_common.index < SPLIT_DATE].copy()
test_common  = weekly_common.loc[weekly_common.index >= SPLIT_DATE].copy()

In [ ]:
sample_summary = pd.DataFrame({
    'sample_start':   [weekly_full.index.min().date(),       weekly_common.index.min().date()],
    'train_start':    [train_full.index.min().date(),        train_common.index.min().date()],
    'train_end':      [train_full.index.max().date(),        train_common.index.max().date()],
    'train_weeks':    [len(train_full),                      len(train_common)],
    'train_years':    [round(len(train_full) / 52, 1),       round(len(train_common) / 52, 1)],
    'test_weeks':     [len(test_full),                       len(test_common)],
    'weeks_lost':     [0,                                    len(train_full) - len(train_common)],
}, index=['full_sample', 'common_sample'])
display(sample_summary)

The common sample training period is approximately 21 years (1997–2018) versus approximately 27.5 years for the full baseline sample. The weeks-lost column quantifies the training history removed solely by requiring high-yield spread availability. Any strategy comparison that uses both samples simultaneously must account for this difference before attributing performance changes to the new variables.

---

## 3. Phase 3 — Feature Transformations

WTI crude oil and the high-yield spread are tested in two forms each, chosen for economic interpretability and minimal data requirements.

**WTI crude oil**
- `wti_log`: log of the weekly WTI price level. Interpretable as a scale-free measure of the oil price environment; coefficient in the fair-value regression is an elasticity.
- `wti_ret`: weekly percentage change in WTI price. Captures short-run oil price momentum, which may influence near-term energy sector sentiment independently of the price level.

**High-yield credit spread**
- `hy_level`: raw spread in percent. Reflects the current level of macro-financial stress. A higher spread compresses risk asset valuations including energy equities.
- `hy_chg`: weekly change in the spread. Captures the direction of credit market movement; a deteriorating credit environment (rising spread) may presage equity weakness independently of the level.

All transformation selection is performed on the training sample only.

---

## 4. Phase 4 & 5 — Feature Specifications and Joint Selection

Nine candidate feature specifications are evaluated. The baseline uses the original two inputs. The remaining eight add one or both new variables in each of their two transformation forms. This design isolates the contribution of each variable and each transformation.

For rolling regression, each specification is evaluated across the full window-and-threshold grid on the common training sample; the reported score is the best training cumulative return over all hyperparameter combinations. The specification with the highest score is selected.

The same procedure is applied independently to the Kalman filter, allowing the two strategies to prefer different specifications.

In [ ]:
def rolling_reg_signals(df, features, window, threshold, min_gap_periods=52):
    log_price = np.log(df['enrs'].values)
    Xmat      = df[features].values
    n         = len(df)
    fitted    = np.full(n, np.nan)
    for t in range(window, n):
        y_w  = log_price[t - window:t]
        X_w  = np.column_stack([np.ones(window), Xmat[t - window:t]])
        beta = np.linalg.lstsq(X_w, y_w, rcond=None)[0]
        fitted[t] = np.r_[1.0, Xmat[t]] @ beta
    gap_s  = pd.Series(log_price - fitted, index=df.index)
    g_mean = gap_s.expanding(min_periods=min_gap_periods).mean()
    g_std  = gap_s.expanding(min_periods=min_gap_periods).std().replace(0, np.nan)
    z_gap  = (gap_s - g_mean) / g_std
    sig    = np.where(z_gap.isna(),       0.0,
             np.where(z_gap < -threshold, 1.0,
             np.where(z_gap >  threshold, -1.0, 0.0)))
    return pd.Series(sig, index=df.index), gap_s, pd.Series(fitted, index=df.index)

def kalman_tvr_signals(df, features, q, r, threshold, min_gap_periods=52):
    log_price = np.log(df['enrs'].values)
    Xmat      = np.column_stack([np.ones(len(df))] + [df[f].values for f in features])
    n, p      = Xmat.shape
    beta      = np.zeros(p)
    P         = np.eye(p) * 10.0
    Q         = np.eye(p) * q
    fitted    = np.full(n, np.nan)
    for t in range(n):
        x_t    = Xmat[t]
        P_pred = P + Q
        y_hat  = float(x_t @ beta)
        S      = float(x_t @ P_pred @ x_t) + r
        K      = (P_pred @ x_t) / S
        beta   = beta + K * (log_price[t] - y_hat)
        P      = (np.eye(p) - np.outer(K, x_t)) @ P_pred
        fitted[t] = y_hat
    gap_s  = pd.Series(log_price - fitted, index=df.index)
    g_mean = gap_s.expanding(min_periods=min_gap_periods).mean()
    g_std  = gap_s.expanding(min_periods=min_gap_periods).std().replace(0, np.nan)
    z_gap  = (gap_s - g_mean) / g_std
    sig    = np.where(z_gap.isna(),       0.0,
             np.where(z_gap < -threshold, 1.0,
             np.where(z_gap >  threshold, -1.0, 0.0)))
    return pd.Series(sig, index=df.index), gap_s, pd.Series(fitted, index=df.index)

def strat_cum_ret(signals, target):
    return ((1 + (signals * target).fillna(0)).prod() - 1) * 100

feature_specs = {
    'baseline':          ['t1y', 'sp500_pe'],
    '+wti_log':          ['t1y', 'sp500_pe', 'wti_log'],
    '+wti_ret':          ['t1y', 'sp500_pe', 'wti_ret'],
    '+hy_level':         ['t1y', 'sp500_pe', 'hy_level'],
    '+hy_chg':           ['t1y', 'sp500_pe', 'hy_chg'],
    '+wti_log+hy_level': ['t1y', 'sp500_pe', 'wti_log', 'hy_level'],
    '+wti_log+hy_chg':   ['t1y', 'sp500_pe', 'wti_log', 'hy_chg'],
    '+wti_ret+hy_level': ['t1y', 'sp500_pe', 'wti_ret', 'hy_level'],
    '+wti_ret+hy_chg':   ['t1y', 'sp500_pe', 'wti_ret', 'hy_chg'],
}

roll_windows    = np.round(np.linspace(26, 260, 20)).astype(int).tolist()
roll_thresholds = np.round(np.linspace(0.25, 2.0, 20), 3).tolist()

In [ ]:
roll_spec_scores = {}
for spec_name, feats in feature_specs.items():
    best_cr = -np.inf
    for win in roll_windows:
        for thr in roll_thresholds:
            sigs, _, _ = rolling_reg_signals(train_common, feats, win, thr)
            cr = strat_cum_ret(sigs, train_common['target'])
            if cr > best_cr:
                best_cr = cr
    roll_spec_scores[spec_name] = best_cr

roll_spec_df = (pd.DataFrame([
    {'spec': k, 'best_train_cum_ret (%)': round(v, 2)}
    for k, v in roll_spec_scores.items()
]).sort_values('best_train_cum_ret (%)', ascending=False).reset_index(drop=True))
display(roll_spec_df)

ROLL_FEATURES = feature_specs[roll_spec_df.iloc[0]['spec']]

In [ ]:
roll_grid = np.full((len(roll_thresholds), len(roll_windows)), np.nan)
for i, thr in enumerate(roll_thresholds):
    for j, win in enumerate(roll_windows):
        sigs, _, _ = rolling_reg_signals(train_common, ROLL_FEATURES, win, thr)
        roll_grid[i, j] = strat_cum_ret(sigs, train_common['target'])

roll_heatmap = pd.DataFrame(roll_grid, index=roll_thresholds, columns=roll_windows)
roll_heatmap.index.name   = 'threshold'
roll_heatmap.columns.name = 'window (weeks)'

bi, bj = np.unravel_index(np.nanargmax(roll_grid), roll_grid.shape)
ROLL_WINDOW    = roll_windows[bj]
ROLL_THRESHOLD = roll_thresholds[bi]

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(roll_heatmap.values, aspect='auto', cmap='RdYlGn', origin='lower')
ax.set_xticks(range(len(roll_windows)));    ax.set_xticklabels(roll_windows)
ax.set_yticks(range(len(roll_thresholds))); ax.set_yticklabels(roll_thresholds)
ax.set_xlabel('Rolling window (weeks)'); ax.set_ylabel('Signal threshold')
ax.set_title(f'Rolling Regression — Selected spec: {roll_spec_df.iloc[0]["spec"]} — Training Cumulative Return (%)',
             fontsize=10, loc='left')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

roll_selected = pd.DataFrame({
    'features':  [', '.join(ROLL_FEATURES)],
    'window':    [ROLL_WINDOW],
    'threshold': [ROLL_THRESHOLD],
}, index=['rolling regression (common)'])
display(roll_selected)

### 4.2 Kalman Filter — Independent Feature Selection

The Kalman strategy independently searches the same nine specifications over the full (q, r, threshold) grid. The reported score for each specification is the best training cumulative return over all 125 hyperparameter combinations. `KF_FEATURES`, `KF_Q`, `KF_R`, and `KF_THRESHOLD` are set from the top-ranked specification.

In [ ]:
kf_q_vals     = np.logspace(-7, -1, 20).tolist()
kf_r_vals     = np.logspace(-1,  1, 20).tolist()
kf_thresholds = np.round(np.linspace(0.25, 2.0, 20), 3).tolist()

kf_spec_scores = {}
for spec_name, feats in feature_specs.items():
    best_cr = -np.inf
    for q_val in kf_q_vals:
        for r_val in kf_r_vals:
            for thr in kf_thresholds:
                sigs, _, _ = kalman_tvr_signals(train_common, feats, q_val, r_val, thr)
                cr = strat_cum_ret(sigs, train_common['target'])
                if cr > best_cr:
                    best_cr = cr
    kf_spec_scores[spec_name] = best_cr

kf_spec_df = (pd.DataFrame([
    {'spec': k, 'best_train_cum_ret (%)': round(v, 2)}
    for k, v in kf_spec_scores.items()
]).sort_values('best_train_cum_ret (%)', ascending=False).reset_index(drop=True))
display(kf_spec_df)

KF_FEATURES = feature_specs[kf_spec_df.iloc[0]['spec']]

In [ ]:
kf_grid = np.full((len(kf_thresholds), len(kf_q_vals), len(kf_r_vals)), np.nan)
for i, thr in enumerate(kf_thresholds):
    for j, q_val in enumerate(kf_q_vals):
        for m, r_val in enumerate(kf_r_vals):
            sigs, _, _ = kalman_tvr_signals(train_common, KF_FEATURES, q_val, r_val, thr)
            kf_grid[i, j, m] = strat_cum_ret(sigs, train_common['target'])

kf_qr_best = kf_grid.max(axis=0)
bi, bj, bm = np.unravel_index(np.nanargmax(kf_grid), kf_grid.shape)
KF_Q         = kf_q_vals[bj]
KF_R         = kf_r_vals[bm]
KF_THRESHOLD = kf_thresholds[bi]

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(kf_qr_best, aspect='auto', cmap='RdYlGn', origin='lower')
ax.set_xticks(range(len(kf_r_vals)));    ax.set_xticklabels(kf_r_vals)
ax.set_yticks(range(len(kf_q_vals)));   ax.set_yticklabels([f'{q:.0e}' for q in kf_q_vals])
ax.set_xlabel('Observation noise R'); ax.set_ylabel('Process noise scale Q')
ax.set_title(f'Kalman Filter — Selected spec: {kf_spec_df.iloc[0]["spec"]} — Best Training Return (%) over threshold',
             fontsize=10, loc='left')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

kf_selected = pd.DataFrame({
    'features':  [', '.join(KF_FEATURES)],
    'Q scale':   [KF_Q],
    'R':         [KF_R],
    'threshold': [KF_THRESHOLD],
}, index=['kalman filter (common)'])
display(kf_selected)

---

## 5. Phase 7 — Model Family Walk-Forward Backtests

Five model families are constructed for each strategy. Each family is independently tuned on its appropriate training sample. Comparing M1 to M2 isolates the sample-truncation effect. Comparing M2 to M3 isolates the WTI contribution. Comparing M2 to M4 isolates the high-yield spread contribution. Comparing M2 to M5 shows the best achievable improvement from the new variables.

| Model | Features | Training sample |
|---|---|---|
| M1 | Baseline [`t1y`, `sp500_pe`] | Full (~27.5 yrs) |
| M2 | Baseline [`t1y`, `sp500_pe`] | Common (~21 yrs) |
| M3 | Baseline + `wti_log` | Common |
| M4 | Baseline + `hy_level` | Common |
| M5 | Selected extended spec | Common |

In [ ]:
def tune_rolling(train_df, feats, windows, thresholds):
    grid = np.full((len(thresholds), len(windows)), np.nan)
    for i, thr in enumerate(thresholds):
        for j, win in enumerate(windows):
            sigs, _, _ = rolling_reg_signals(train_df, feats, win, thr)
            grid[i, j] = strat_cum_ret(sigs, train_df['target'])
    bi, bj = np.unravel_index(np.nanargmax(grid), grid.shape)
    return windows[bj], thresholds[bi]

def tune_kalman(train_df, feats, q_vals, r_vals, thresholds):
    grid = np.full((len(thresholds), len(q_vals), len(r_vals)), np.nan)
    for i, thr in enumerate(thresholds):
        for j, q in enumerate(q_vals):
            for m, r in enumerate(r_vals):
                sigs, _, _ = kalman_tvr_signals(train_df, feats, q, r, thr)
                grid[i, j, m] = strat_cum_ret(sigs, train_df['target'])
    bi, bj, bm = np.unravel_index(np.nanargmax(grid), grid.shape)
    return q_vals[bj], r_vals[bm], thresholds[bi]

BASELINE_FEATS = ['t1y', 'sp500_pe']
WTI_FEATS      = ['t1y', 'sp500_pe', 'wti_log']
HY_FEATS       = ['t1y', 'sp500_pe', 'hy_level']

In [ ]:
m1_win, m1_thr = tune_rolling(train_full,   BASELINE_FEATS, roll_windows, roll_thresholds)
m2_win, m2_thr = tune_rolling(train_common, BASELINE_FEATS, roll_windows, roll_thresholds)
m3_win, m3_thr = tune_rolling(train_common, WTI_FEATS,      roll_windows, roll_thresholds)
m4_win, m4_thr = tune_rolling(train_common, HY_FEATS,       roll_windows, roll_thresholds)

roll_sigs_m1 = rolling_reg_signals(weekly_full,   BASELINE_FEATS, m1_win, m1_thr)[0]
roll_sigs_m2 = rolling_reg_signals(weekly_common, BASELINE_FEATS, m2_win, m2_thr)[0]
roll_sigs_m3 = rolling_reg_signals(weekly_common, WTI_FEATS,      m3_win, m3_thr)[0]
roll_sigs_m4 = rolling_reg_signals(weekly_common, HY_FEATS,       m4_win, m4_thr)[0]
roll_sigs_m5 = rolling_reg_signals(weekly_common, ROLL_FEATURES,  ROLL_WINDOW, ROLL_THRESHOLD)[0]

model_params_roll = pd.DataFrame({
    'features':   [', '.join(BASELINE_FEATS), ', '.join(BASELINE_FEATS),
                   ', '.join(WTI_FEATS), ', '.join(HY_FEATS), ', '.join(ROLL_FEATURES)],
    'sample':     ['full', 'common', 'common', 'common', 'common'],
    'window':     [m1_win, m2_win, m3_win, m4_win, ROLL_WINDOW],
    'threshold':  [m1_thr, m2_thr, m3_thr, m4_thr, ROLL_THRESHOLD],
}, index=['M1 roll', 'M2 roll', 'M3 roll (+wti)', 'M4 roll (+hy)', 'M5 roll (selected)'])
display(model_params_roll)

In [ ]:
mk1_q, mk1_r, mk1_thr = tune_kalman(train_full,   BASELINE_FEATS, kf_q_vals, kf_r_vals, kf_thresholds)
mk2_q, mk2_r, mk2_thr = tune_kalman(train_common, BASELINE_FEATS, kf_q_vals, kf_r_vals, kf_thresholds)
mk3_q, mk3_r, mk3_thr = tune_kalman(train_common, WTI_FEATS,      kf_q_vals, kf_r_vals, kf_thresholds)
mk4_q, mk4_r, mk4_thr = tune_kalman(train_common, HY_FEATS,       kf_q_vals, kf_r_vals, kf_thresholds)

kf_sigs_mk1 = kalman_tvr_signals(weekly_full,   BASELINE_FEATS, mk1_q, mk1_r, mk1_thr)[0]
kf_sigs_mk2 = kalman_tvr_signals(weekly_common, BASELINE_FEATS, mk2_q, mk2_r, mk2_thr)[0]
kf_sigs_mk3 = kalman_tvr_signals(weekly_common, WTI_FEATS,      mk3_q, mk3_r, mk3_thr)[0]
kf_sigs_mk4 = kalman_tvr_signals(weekly_common, HY_FEATS,       mk4_q, mk4_r, mk4_thr)[0]
kf_sigs_mk5 = kalman_tvr_signals(weekly_common, KF_FEATURES,    KF_Q,  KF_R,  KF_THRESHOLD)[0]

model_params_kf = pd.DataFrame({
    'features':   [', '.join(BASELINE_FEATS), ', '.join(BASELINE_FEATS),
                   ', '.join(WTI_FEATS), ', '.join(HY_FEATS), ', '.join(KF_FEATURES)],
    'sample':     ['full', 'common', 'common', 'common', 'common'],
    'Q scale':    [mk1_q, mk2_q, mk3_q, mk4_q, KF_Q],
    'R':          [mk1_r, mk2_r, mk3_r, mk4_r, KF_R],
    'threshold':  [mk1_thr, mk2_thr, mk3_thr, mk4_thr, KF_THRESHOLD],
}, index=['MK1', 'MK2', 'MK3 (+wti)', 'MK4 (+hy)', 'MK5 (selected)'])
display(model_params_kf)

---

## 6. Phase 8 — Results and Diagnosis

### 6.1 Performance Metrics

Performance is evaluated over the common test period (January 2018 – February 2026). The M1 baseline uses the full-sample model but is evaluated only on the overlapping test window for a fair comparison.

The risk-free rate is the average 1-year Treasury yield over the test period converted to a weekly equivalent.

In [ ]:
test_dates_full   = weekly_full.index[weekly_full.index >= SPLIT_DATE]
test_dates_common = weekly_common.index[weekly_common.index >= SPLIT_DATE]
test_dates        = test_dates_common

rf_annual = weekly_common.loc[test_dates, 't1y'].mean() / 100
rf_weekly = rf_annual / 52

def ann_return(r, freq=52):
    r = r.dropna()
    return (1 + r).prod() ** (freq / len(r)) - 1 if len(r) > 0 else np.nan

def ann_vol(r, freq=52):
    return r.dropna().std() * np.sqrt(freq)

def sharpe(r, rf, freq=52):
    ex = r.dropna() - rf
    return ex.mean() / ex.std() * np.sqrt(freq) if ex.std() != 0 else np.nan

def max_dd(r):
    w = (1 + r.fillna(0)).cumprod()
    return ((w - w.cummax()) / w.cummax()).min()

def hit_rate(sig, tgt):
    mask = (sig != 0) & tgt.notna()
    if mask.sum() == 0: return np.nan
    return ((sig[mask] * tgt[mask]) > 0).sum() / mask.sum()

def gini(r):
    a = np.sort(np.abs(r.dropna()))
    n = len(a)
    if n == 0 or a.sum() == 0: return np.nan
    return (2 * (np.arange(1, n + 1) * a).sum()) / (n * a.sum()) - (n + 1) / n

tgt_common = weekly_common.loc[test_dates, 'target']
tgt_full_on_common = weekly_full.loc[weekly_full.index.isin(test_dates), 'target']

def make_ret(sigs, tgt_series, test_idx):
    s = sigs.loc[sigs.index.isin(test_idx)]
    t = tgt_series.loc[tgt_series.index.isin(test_idx)]
    s, t = s.align(t, join='inner')
    return (s * t).fillna(0)

ret_m1 = make_ret(roll_sigs_m1, weekly_full['target'],   test_dates)
ret_m2 = make_ret(roll_sigs_m2, weekly_common['target'], test_dates)
ret_m3 = make_ret(roll_sigs_m3, weekly_common['target'], test_dates)
ret_m4 = make_ret(roll_sigs_m4, weekly_common['target'], test_dates)
ret_m5 = make_ret(roll_sigs_m5, weekly_common['target'], test_dates)

ret_mk1 = make_ret(kf_sigs_mk1, weekly_full['target'],   test_dates)
ret_mk2 = make_ret(kf_sigs_mk2, weekly_common['target'], test_dates)
ret_mk3 = make_ret(kf_sigs_mk3, weekly_common['target'], test_dates)
ret_mk4 = make_ret(kf_sigs_mk4, weekly_common['target'], test_dates)
ret_mk5 = make_ret(kf_sigs_mk5, weekly_common['target'], test_dates)

ret_bh    = weekly_common.loc[test_dates, 'target'].fillna(0)
ret_sp500 = weekly_common.loc[test_dates, 'sp500_target'].fillna(0)

In [ ]:
sig_lookup = {
    'M1 Roll (baseline, full)':      roll_sigs_m1.reindex(test_dates).fillna(0),
    'M2 Roll (baseline, common)':    roll_sigs_m2.reindex(test_dates).fillna(0),
    'M3 Roll (+wti_log, common)':    roll_sigs_m3.reindex(test_dates).fillna(0),
    'M4 Roll (+hy_level, common)':   roll_sigs_m4.reindex(test_dates).fillna(0),
    'M5 Roll (selected, common)':    roll_sigs_m5.reindex(test_dates).fillna(0),
    'MK1 KF (baseline, full)':       kf_sigs_mk1.reindex(test_dates).fillna(0),
    'MK2 KF (baseline, common)':     kf_sigs_mk2.reindex(test_dates).fillna(0),
    'MK3 KF (+wti_log, common)':     kf_sigs_mk3.reindex(test_dates).fillna(0),
    'MK4 KF (+hy_level, common)':    kf_sigs_mk4.reindex(test_dates).fillna(0),
    'MK5 KF (selected, common)':     kf_sigs_mk5.reindex(test_dates).fillna(0),
    'BnH S5ENRS':                    pd.Series(1.0, index=test_dates),
    'S&P 500':                       pd.Series(1.0, index=test_dates),
}

ret_lookup = {
    'M1 Roll (baseline, full)':      ret_m1,
    'M2 Roll (baseline, common)':    ret_m2,
    'M3 Roll (+wti_log, common)':    ret_m3,
    'M4 Roll (+hy_level, common)':   ret_m4,
    'M5 Roll (selected, common)':    ret_m5,
    'MK1 KF (baseline, full)':       ret_mk1,
    'MK2 KF (baseline, common)':     ret_mk2,
    'MK3 KF (+wti_log, common)':     ret_mk3,
    'MK4 KF (+hy_level, common)':    ret_mk4,
    'MK5 KF (selected, common)':     ret_mk5,
    'BnH S5ENRS':                    ret_bh,
    'S&P 500':                       ret_sp500,
}

tgt_ref = weekly_common.loc[test_dates, 'target']

perf = pd.DataFrame({
    name: {
        'Ann. Return (%)':  round(ann_return(r) * 100, 2),
        'Ann. Vol (%)':     round(ann_vol(r) * 100, 2),
        'Sharpe':           round(sharpe(r, rf_weekly), 3),
        'Max DD (%)':       round(max_dd(r) * 100, 2),
        'Hit Rate (%)':     round(hit_rate(sig_lookup[name], tgt_ref) * 100, 2),
        'Gini':             round(gini(r), 3),
    }
    for name, r in ret_lookup.items()
}).T

display(perf)

### 6.2 Cumulative Return Comparison

Two plots separate the rolling regression family from the Kalman family. Both include the two benchmarks for reference.

In [ ]:
roll_names  = ['M1 Roll (baseline, full)', 'M2 Roll (baseline, common)',
               'M3 Roll (+wti_log, common)', 'M4 Roll (+hy_level, common)',
               'M5 Roll (selected, common)', 'BnH S5ENRS', 'S&P 500']
roll_colors = ['steelblue', 'royalblue', 'seagreen', 'mediumpurple',
               'firebrick', 'darkorange', 'black']

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

for name, col in zip(roll_names, roll_colors):
    r = ret_lookup[name]
    cum = (1 + r.fillna(0)).cumprod()
    axes[0].plot(cum.index, cum.values, linewidth=0.9, color=col, label=name)
axes[0].axhline(1, color='grey', linewidth=0.4, linestyle='--')
axes[0].set_ylabel('Growth of $1')
axes[0].set_title('Rolling Regression Model Families — Out-of-Sample', fontsize=11, loc='left')
axes[0].legend(fontsize=7, ncol=2)

kf_names  = ['MK1 KF (baseline, full)', 'MK2 KF (baseline, common)',
             'MK3 KF (+wti_log, common)', 'MK4 KF (+hy_level, common)',
             'MK5 KF (selected, common)', 'BnH S5ENRS', 'S&P 500']
kf_colors = ['steelblue', 'royalblue', 'seagreen', 'mediumpurple',
             'firebrick', 'darkorange', 'black']

for name, col in zip(kf_names, kf_colors):
    r = ret_lookup[name]
    cum = (1 + r.fillna(0)).cumprod()
    axes[1].plot(cum.index, cum.values, linewidth=0.9, color=col, label=name)
axes[1].axhline(1, color='grey', linewidth=0.4, linestyle='--')
axes[1].set_ylabel('Growth of $1')
axes[1].set_title('Kalman Filter Model Families — Out-of-Sample', fontsize=11, loc='left')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

### 6.3 Diagnosis

The table and plots above answer the seven diagnostic questions directly.

**Q1 — How much of the performance change is due to sample truncation?**  
Compare M1 (baseline, full sample) versus M2 (baseline, common sample). Any difference in performance between these two models is attributable entirely to the shorter training period, not to the new variables.

**Q2 — Does WTI crude help on its own?**  
Compare M3 (baseline + wti_log) versus M2 (baseline on same common sample). A better Sharpe or annualised return in M3 would indicate a positive WTI contribution.

**Q3 — Does high-yield spread help on its own?**  
Compare M4 (baseline + hy_level) versus M2. Same logic.

**Q4 — Do both new variables help together?**  
M5 carries the jointly selected specification. If M5 uses an extended spec, compare its test performance against M2.

**Q5 — Does the answer differ between strategies?**  
The rolling regression (M1–M5) and Kalman (MK1–MK5) family selections may differ because feature selection was performed independently for each strategy. The spec selection tables above show the ranked specifications for each.

**Q6 — Which transformations work best?**  
The spec selection tables rank all nine specifications including each transformation form. The highest-ranked single-variable extensions identify the better transformation for each new variable.

**Q7 — Were the added variables genuinely unhelpful, or was the prior implementation too rigid?**  
The prior implementation used one fixed configuration for pre-screening and forced both strategies to share a feature set. This diagnostic notebook uses joint search and independent selection. If the new variables still do not rank above the baseline after this more thorough search, the evidence is stronger that they are genuinely unhelpful for this asset over this sample.

---

## 7. Conclusions

The diagnostic analysis separates four distinct effects that were conflated in the initial extension:
1. Sample truncation from the high-yield spread start date
2. The marginal contribution of WTI crude oil
3. The marginal contribution of the high-yield credit spread
4. The choice of variable transformation

Whether the new fundamentals improve or worsen forecasting performance is determined by the training-sample selection results above. If the spec selection tables rank `baseline` first for both strategies, the new variables offer no training-period improvement even after correcting for the rigidity of the earlier implementation. If extended specifications rank higher, the improved tuning procedure has recovered value that the original extension missed.

The rolling regression and Kalman strategies are allowed to disagree, which is appropriate since they exploit the fundamental-price relationship through different dynamics. Any divergence between their selected specifications reflects genuinely different sensitivity to the new inputs.

---

## What Remains

- Verify that out-of-sample results are consistent with training-period rankings before drawing conclusions for the written report
- Consider whether the high-yield spread data start date (~Jan 1997) introduces a survivorship-like bias in the common-sample comparison
- Incorporate diagnostic findings into the narrative of the final project report